# FIRST-PL Optimal Extraction → Coupling Map

This tutorial demonstrates **FIRSTPL extraction** — the full 2D optimal extractor
specific to the SCExAO/FIRST-PL instrument.

### FIRSTPL vs trace extraction

| | `trace_box` / `simple_box` | `FIRSTPL` |
|---|---|---|
| Profile model | Rectangular aperture | Full 2D sparse matrix from `visPLred` calibration |
| Calibration file | `traces.npz` | `model.npz` |
| Cross-talk rejection | None | Yes — deblends overlapping fiber PSFs |
| Wavelength solution | Optional (`traces.npz`) | Baked into `model.npz` |
| Speed | ~ms per image | ~100 ms per image |

### Prerequisites

- `average_map.h5` — output of `plred-average` (Step 4). [Run `pipeline_end_to_end.ipynb` first if needed.]
- `model.npz` — spectrum model built by `visPLred` (`SpectrumModel`). See `visPLred/tutorials/pre2_spectrum_model.ipynb`.

### Outline

1. [Inspect the model file](#1-model)
2. [Check ROI overlap](#2-overlap)
3. [Build and test the extractor](#3-extractor)
4. [Write the config and run extraction](#4-run)
5. [Inspect the coupling map](#5-inspect)
6. [Compare with trace extraction](#6-compare)
7. [CLI reference](#7-cli)

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import h5py
from astropy.io import fits
from configobj import ConfigObj

import PLred.specextract as specextract

In [ ]:
# ─── Input files ────────────────────────────────────────────────────────────
AVERAGE_MAP = 'timestamp_matching_output/average_map.h5'
MODEL_FILE  = 'timestamp_matching_output/model.npz'

# ─── Output ─────────────────────────────────────────────────────────────────
CONFIG_INI  = 'timestamp_matching_output/obs_firstpl.ini'
OUTPUT_FITS = 'timestamp_matching_output/coupling_map_firstpl.fits'

# ─── Read ROI and grid from the averaged H5 ─────────────────────────────────
with h5py.File(AVERAGE_MAP, 'r') as f:
    PLCAM_ROI = tuple(int(v) for v in f['metadata/plcam_roi'][:])
    map_n     = int(f.attrs['map_n'])
    x_mas     = f['metadata/x_mas'][:]
    y_mas     = f['metadata/y_mas'][:]

print(f'average_map.h5 map_n  = {map_n}')
print(f'plcam_roi             = {PLCAM_ROI}   (y0,y1,x0,x1 in detector coords)')
print(f'stored image size     = {PLCAM_ROI[1]-PLCAM_ROI[0]} × {PLCAM_ROI[3]-PLCAM_ROI[2]} pixels')

---
<a id='1-model'></a>
## 1. Inspect the model file

`model.npz` is produced by `visPLred.SpectrumModel` from a neon lamp calibration.
It contains a sparse 2D extraction matrix and an optional wavelength solution.

In [ ]:
d = np.load(MODEL_FILE, allow_pickle=False)
print('Keys:', list(d.keys()))
print()

xmin_det  = int(d['xmin'])      # model spectral range — detector column coords
xmax_det  = int(d['xmax'])
ny_full   = int(d['ny_full'])   # full detector height
has_wav   = 'wav_map' in d

# Matrix stored as CSR
shape = tuple(d['matrix_shape'])
nwav_model = xmax_det - xmin_det
nfib = shape[0] // nwav_model

print(f'Model spectral range : detector x = [{xmin_det}, {xmax_det})   nwav = {nwav_model}')
print(f'Full detector height : ny_full = {ny_full}')
print(f'Matrix shape         : {shape}   → nfib = {nfib}')
print(f'Wavelength solution  : {has_wav}')

if has_wav:
    wav = d['wav_map']
    print(f'  wav_map shape : {wav.shape}   range = {wav.min():.2f} – {wav.max():.2f} nm')

---
<a id='2-overlap'></a>
## 2. Check ROI overlap

The model was built on full-detector frames.  
The averaged H5 stores only the hardware-ROI crop `x = [x0, x1)`.  
`make_FIRSTPL_extractor` automatically trims the matrix to the overlap — let's compute it manually first.

In [ ]:
roi_y0, roi_y1, roi_x0, roi_x1 = PLCAM_ROI

eff_xmin = max(xmin_det, roi_x0)
eff_xmax = min(xmax_det, roi_x1)
nwav_eff = eff_xmax - eff_xmin

print(f'ROI x range    : [{roi_x0}, {roi_x1})   width = {roi_x1 - roi_x0} px')
print(f'Model x range  : [{xmin_det}, {xmax_det})   width = {nwav_model} px')
print(f'Overlap        : [{eff_xmin}, {eff_xmax})   → effective nwav = {nwav_eff}')
print()

if nwav_eff <= 0:
    print('ERROR: no overlap between model and ROI — check plcam_roi and model xmin/xmax')
elif nwav_eff < nwav_model:
    lost = 100 * (1 - nwav_eff / nwav_model)
    print(f'ROI covers {nwav_eff}/{nwav_model} model channels ({lost:.0f}% trimmed).')
    print('Extraction will use only the overlapping columns.')
else:
    print('Full model range is within the ROI — no trimming needed.')

# Visualise
fig, ax = plt.subplots(figsize=(9, 1.4))
ax.barh(0, xmax_det - xmin_det, left=xmin_det, height=0.4,
        color='steelblue', alpha=0.7, label=f'model [{xmin_det},{xmax_det})')
ax.barh(0, roi_x1 - roi_x0,   left=roi_x0,   height=0.4,
        color='tomato',    alpha=0.5, label=f'ROI [{roi_x0},{roi_x1})')
ax.barh(0, nwav_eff,           left=eff_xmin,  height=0.4,
        color='gold',      alpha=0.9, label=f'overlap [{eff_xmin},{eff_xmax})')
ax.set_xlabel('Detector x column')
ax.set_yticks([])
ax.set_title('Model vs ROI coverage')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

---
<a id='3-extractor'></a>
## 3. Build and test the extractor

`make_FIRSTPL_extractor` loads the sparse matrix, trims it to the ROI overlap,
and returns a callable `f(image) → (nfib, nwav_effective)`.

In [ ]:
extractor = specextract.make_FIRSTPL_extractor(
    model_file = MODEL_FILE,
    plcam_roi  = PLCAM_ROI,
    dark       = None,       # dark subtraction was already done during ingest
    var_const  = 200,        # regularisation constant
    thresh     = 0.1,        # damping threshold
)

info = extractor._info
print(f'Extractor type      : {info["extractor"]}')
print(f'Effective x range   : [{info["xmin"]}, {info["xmax"]})')
print(f'plcam_roi           : {info["plcam_roi"]}')
print(f'Wavelength solution : {info["has_wavmap"]}')
print(f'Output nwav         : {info["xmax"] - info["xmin"]}')

In [ ]:
# Test on the brightest spatial bin
with h5py.File(AVERAGE_MAP, 'r') as f:
    avg_plcam = f['avg_PLcam'][:]          # (map_n, map_n, ny, nx)
    nframes   = f['metadata/nframes'][:]   # (map_n, map_n)

ix, iy = np.unravel_index(np.argmax(nframes), nframes.shape)
ref_image = avg_plcam[ix, iy]             # (ny, nx) — the raw ROI image

spec = extractor(ref_image)               # → (nfib, nwav_effective)
print(f'Reference bin        : ({ix},{iy}),  nframes = {nframes[ix,iy]}')
print(f'Extractor output     : shape = {spec.shape}   (nfib={spec.shape[0]}, nwav={spec.shape[1]})')

# Wavelength axis
if info['has_wavmap']:
    wm = np.load(MODEL_FILE, allow_pickle=False)['wav_map']
    xs = info['xmin'] - xmin_det
    xe = info['xmax'] - xmin_det
    wav_axis = wm[0, xs:xe]    # use fiber 0 as reference
    xlabel = 'Wavelength (nm)'
else:
    wav_axis = np.arange(spec.shape[1])
    xlabel = 'Spectral channel'

# Plot a few fibers
fig, ax = plt.subplots(figsize=(9, 3.5))
for fi in range(min(8, nfib)):
    ax.plot(wav_axis, spec[fi], lw=1.2, label=f'fiber {fi}')
ax.set_xlabel(xlabel)
ax.set_ylabel('Counts (optimal extraction)')
ax.set_title(f'FIRSTPL extraction — bin ({ix},{iy})  nframes={nframes[ix,iy]}')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# All fibers as a heatmap
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(spec, aspect='auto', origin='upper',
               extent=[wav_axis[0], wav_axis[-1], nfib, 0],
               cmap='viridis')
plt.colorbar(im, ax=ax, label='Counts')
ax.set_xlabel(xlabel)
ax.set_ylabel('Fiber index')
ax.set_title(f'All {nfib} fibers — FIRSTPL extraction  bin ({ix},{iy})')
plt.tight_layout()
plt.show()

---
<a id='4-run'></a>
## 4. Write the config and run extraction

The config uses `extractor = FIRSTPL` and points `model_file` at the `model.npz`.
All other pipeline steps stay the same.

In [ ]:
cfg = ConfigObj()
cfg.filename = CONFIG_INI

cfg['Specextract'] = {
    'input'      : AVERAGE_MAP,
    'extractor'  : 'FIRSTPL',
    'output'     : OUTPUT_FITS,
    'plcam_roi'  : ','.join(str(v) for v in PLCAM_ROI),
    'model_file' : MODEL_FILE,
    'var_const'  : '200',
    'thresh'     : '0.1',
    'nonlin_file': '',
    'truncate':   '0',   # remove first/last N channels (edges unstable for FIRSTPL)
    # not used for FIRSTPL:
    'trace_file'  : '',
    'profile_file': '',
}

cfg.write()
print(f'Config written to: {CONFIG_INI}')
print()
with open(CONFIG_INI) as fh:
    print(fh.read())

In [ ]:
%%time
specextract.extract_from_config(CONFIG_INI)
print(f'\nOutput: {OUTPUT_FITS}')

**CLI equivalent:**
```bash
plred-extract timestamp_matching_output/obs_firstpl.ini
```

---
<a id='5-inspect'></a>
## 5. Inspect the coupling map

In [ ]:
with fits.open(OUTPUT_FITS) as hdul:
    hdul.info()
    print()
    hdr  = hdul[0].header
    data = hdul[0].data     # (map_n, map_n, nfib, nwav)
    nfr  = hdul[1].data     # (map_n, map_n)

print(f'Shape  : {data.shape}  (map_n={map_n}, nfib={data.shape[2]}, nwav={data.shape[3]})')
print(f'Extractor : {hdr.get("HIERARCH SPEX TYPE", "n/a")}')
print(f'Valid bins: {int((nfr >= 1).sum())} / {map_n*map_n}')
print(f'nframes per bin:')
print(nfr)

In [ ]:
# Mean coupling map per fiber (average over spectral channels)
mean_map = np.nanmean(data, axis=3)   # (map_n, map_n, nfib)

ncols = 8
nrows = int(np.ceil(nfib / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))
axes = axes.flatten()
extent = [x_mas[0], x_mas[-1], y_mas[-1], y_mas[0]]

for fi in range(nfib):
    ax  = axes[fi]
    m   = mean_map[:, :, fi]
    pos = m[m > 0]
    vmax = np.nanpercentile(pos, 98) if len(pos) else 1.0
    ax.imshow(m, extent=extent, origin='upper',
              cmap='inferno', vmin=0, vmax=max(vmax, 1e-9), aspect='equal')
    ax.set_title(f'fib {fi}', fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

for ax in axes[nfib:]:
    ax.set_visible(False)

fig.suptitle('Mean coupling maps — FIRSTPL extraction', y=1.01)
fig.text(0.5, -0.01, 'x (mas)', ha='center')
fig.text(-0.01, 0.5, 'y (mas)', va='center', rotation='vertical')
plt.tight_layout()
plt.show()

In [ ]:
# Spectra of the most-populated bin on wavelength axis
ix, iy = np.unravel_index(np.argmax(nfr), nfr.shape)
spec_cm = data[ix, iy]   # (nfib, nwav)

fig, ax = plt.subplots(figsize=(9, 3.5))
for fi in range(nfib):
    ax.plot(wav_axis, spec_cm[fi], lw=0.8, alpha=0.7, color=f'C{fi % 10}')
ax.set_xlabel(xlabel)
ax.set_ylabel('Counts')
ax.set_title(f'Coupling map spectra — bin ({ix},{iy})  nframes={nfr[ix,iy]}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id='6-compare'></a>
## 6. Compare with trace extraction

Load the trace-extracted coupling map (from `tutorial_trace_extraction.ipynb`) and
overlay spectra for the same bin and fiber.

In [ ]:
import os

trace_fits = 'timestamp_matching_output/coupling_map_trace.fits'

if not os.path.exists(trace_fits):
    print(f'{trace_fits} not found — run tutorial_trace_extraction.ipynb first.')
else:
    with fits.open(trace_fits) as hdul:
        data_trace = hdul[0].data   # (map_n, map_n, nfib, nwav_trace)

    fib_i = 0
    spec_fp = data[ix, iy, fib_i]          # FIRSTPL
    spec_tr = data_trace[ix, iy, fib_i]    # trace_box
    x_tr    = np.arange(spec_tr.shape[0])  # trace uses pixel coords

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

    # Left: overlay on respective x-axes
    axes[0].plot(wav_axis, spec_fp,  lw=1.5, label='FIRSTPL (wavelength)')
    ax2 = axes[0].twiny()
    ax2.plot(x_tr,     spec_tr, lw=1.0, ls='--', color='tomato',
             label='trace_box (pixel)')
    axes[0].set_xlabel('Wavelength (nm)')
    ax2.set_xlabel('Spectral channel (pixel)', color='tomato')
    axes[0].set_ylabel('Counts')
    axes[0].set_title(f'Fiber {fib_i} — bin ({ix},{iy})')
    axes[0].legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)
    axes[0].grid(alpha=0.3)

    # Right: mean coupling map for fiber 0
    kw = dict(extent=[x_mas[0], x_mas[-1], y_mas[-1], y_mas[0]],
              origin='upper', cmap='inferno', aspect='equal')
    m_fp  = mean_map[:, :, fib_i]
    m_tr  = np.nanmean(data_trace, axis=3)[:, :, fib_i]
    vmax  = max(np.nanmax(m_fp[m_fp > 0]) if np.any(m_fp > 0) else 1,
                np.nanmax(m_tr[m_tr > 0]) if np.any(m_tr > 0) else 1)
    axes[1].imshow(m_fp,  vmin=0, vmax=vmax, **kw)
    axes[1].set_title(f'FIRSTPL  vs  trace_box — fiber {fib_i} coupling map')
    axes[1].set_xlabel('x (mas)'); axes[1].set_ylabel('y (mas)')

    print('FIRSTPL   total counts (brightest bin, fiber 0):', spec_fp.sum())
    print('trace_box total counts (brightest bin, fiber 0):', spec_tr.sum())

    plt.tight_layout()
    plt.show()

---
<a id='7-cli'></a>
## 7. CLI reference

```bash
# Run FIRSTPL extraction with the config written above
plred-extract timestamp_matching_output/obs_firstpl.ini
```

To add FIRSTPL extraction to the main `obs.ini` used by the full pipeline, change:

```ini
[Specextract]
extractor  = FIRSTPL
model_file = timestamp_matching_output/model.npz
var_const  = 200
thresh     = 0.1
nonlin_file =
```

Then `plred-extract obs.ini` (or `plred-run obs.ini --steps 5`) uses the FIRSTPL extractor.

---

**Next step:** load the coupling map into `CouplingMapModel` and run image reconstruction:

```python
from PLred.mapmodel import CouplingMapModel
model = CouplingMapModel(
    mapdata='timestamp_matching_output/coupling_map_firstpl.fits',
    min_nframes=2,
)
```

See **[Step 3: Image reconstruction](step3_image_reconstruction.ipynb)**.